# **<font color="red">Generate Videos with Veo 3.1 in Gemini API</font>**
***Text-to-Video Generation***
- **Veo 3.1** is Google's state-of-the-art model for generating high-fidelity, 8-second 720p, 1080p or 4k video featuring stunning realism and natively generated audio.
- You can access this model programmatically using the Gemini API.
- **Veo 3.1** excels at a wide range of visual and cinematic styles and introduces several new capabilities:
  - **Portrait Videos:** Choose between landscape(***16:9***) and portrait(***9:16***) videos.
  - **Video Extension:** Extend videos that were previously generated using the Veo.
  - **Frame-Specific Generation:** Generate a video by specifying the first and/or last frames.
  - **Image-Based Direction:** Use up to three reference images to guide the content of your generated video.

## **<font color="blue">Dialogue & Sound Effects</font>**

### **Generate Video**

In [ ]:
import time
from google import genai
from google.genai import types

client = genai.Client()

prompt = """A close up of two people staring at a cryptic drawing on a wall, torchlight flickering.
A man murmurs, 'This must be it. That's the secret code.' The woman looks at him and whispering excitedly, 'What did you find?'"""

operation = client.models.generate_videos(
    model="veo-3.1-generate-preview",
    prompt=prompt,
)

# Poll the operation status until the video is ready.
while not operation.done:
    print("Waiting for video generation to complete...")
    time.sleep(10)
    operation = client.operations.get(operation)

# Download the generated video.
generated_video = operation.response.generated_videos[0]
client.files.download(file=generated_video.video)
generated_video.video.save("dialogue_example.mp4")
print("Generated video saved to dialogue_example.mp4")


### **Control the aspect ratio**

In [11]:
# control the aspect ration
import time
from google import genai
from google.genai import types

client = genai.Client()

prompt = """A montage of pizza making: a chef tossing and flattening the floury dough, ladling rich red tomato sauce in a spiral, sprinkling mozzarella cheese and pepperoni, and a final shot of the bubbling golden-brown pizza, upbeat electronic music with a rhythmical beat is playing, high energy professional video."""

operation = client.models.generate_videos(
    model="veo-3.1-generate-preview",
    prompt=prompt,
    config=types.GenerateVideosConfig(
      aspect_ratio="9:16",
    ),
)

# Poll the operation status until the video is ready.
while not operation.done:
    print("Waiting for video generation to complete...")
    time.sleep(10)
    operation = client.operations.get(operation)

# Download the generated video.
generated_video = operation.response.generated_videos[0]
client.files.download(file=generated_video.video)
generated_video.video.save("pizza_making.mp4")
print("Generated video saved to pizza_making.mp4")

### **Control the resolution**

In [ ]:
# control the resolution
import time
from google import genai
from google.genai import types

client = genai.Client()

prompt = """A stunning drone view of the Grand Canyon during a flamboyant sunset that highlights the canyon's colors. The drone slowly flies towards the sun then accelerates, dives and flies inside the canyon."""

operation = client.models.generate_videos(
    model="veo-3.1-generate-preview",
    prompt=prompt,
    config=types.GenerateVideosConfig(
      resolution="4k",
    ),
)

# Poll the operation status until the video is ready.
while not operation.done:
    print("Waiting for video generation to complete...")
    time.sleep(10)
    operation = client.operations.get(operation)

# Download the generated video.
generated_video = operation.response.generated_videos[0]
client.files.download(file=generated_video.video)
generated_video.video.save("4k_grand_canyon.mp4")
print("Generated video saved to 4k_grand_canyon.mp4")

### **Full Script**

In [4]:
"""
Dynamic Brand Introduction Video Generation Pipeline
Mode: Advanced Deep Research + Dialogue & Sound Effects (Veo 3.1)
User-Controlled Length & Language
"""

import os
import time
import asyncio
from datetime import datetime

from google import genai
from google.genai import types
from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.tools import FunctionTool
from google.adk.tools.google_search_tool import google_search
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner

from config import config

# ============================================================
# CONFIG
# ============================================================

os.environ["GOOGLE_API_KEY"] = config.GOOGLE_API_KEY

APP_NAME = "Dynamic-Brand-Dialogue-Video"
USER_ID = "user_1"
SESSION_ID = "session_1"

TEXT_MODEL = "gemini-2.0-flash"
VIDEO_MODEL = "veo-3.1-generate-preview"

# ============================================================
# VIDEO GENERATION TOOL
# ============================================================

async def generate_dialogue_video_tool(prompt: str, brand_name: str) -> dict:
    """
    Generate a dialogue-based brand video using Veo 3.1.
    The brand_name is passed directly from the agent output.
    """
    client = genai.Client()

    print("\n🎬 Generating Dialogue & Sound Effects Brand Video...\n")

    operation = client.models.generate_videos(
        model=VIDEO_MODEL,
        prompt=prompt,
    )

    while not operation.done:
        print("⏳ Waiting for video generation...")
        time.sleep(10)
        operation = client.operations.get(operation)

    generated_video = operation.response.generated_videos[0]

    # Safe filename without regex
    brand_name_clean = "".join(c if c.isalnum() else "_" for c in brand_name.strip())
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Ensure directory exists
    os.makedirs("generated_videos", exist_ok=True)
    filename = os.path.join("generated_videos", f"{brand_name_clean}_{timestamp}.mp4")

    # Save video
    client.files.download(file=generated_video.video)
    generated_video.video.save(filename)

    print(f"\n✅ Video Generated Successfully: {filename}\n")

    return {
        "status": "success",
        "filename": filename,
    }


generate_dialogue_video = FunctionTool(func=generate_dialogue_video_tool)

# ============================================================
# AGENT 1 — DEEP MULTI-SEARCH RESEARCH AGENT
# ============================================================

research_agent = LlmAgent(
    name="deep_brand_research_agent",
    model=TEXT_MODEL,
    instruction="""
You are an elite investigative brand intelligence analyst.

You MUST perform EXTENSIVE research using multiple Google searches.

Perform at least 12–15 search queries including:

- Official website
- About page
- Product pages
- Services
- Leadership team
- Founder interviews
- Press releases
- Media coverage
- Customer testimonials
- Reviews
- Competitors
- Mission & Vision
- Brand identity assets
- Industry reports
- Social media platforms

Collect large, detailed content before summarizing.

Cross-check information from multiple sources.
Ensure accuracy and depth.

--------------------------------------------------

Extract in detail:

Official Brand Name:
Founded Year:
Founder(s):
Headquarters:
Industry:
Website:

Products/Services:
USP:
Market Positioning:
Revenue Model (if available):

Mission:
Vision:
Core Values:
Brand Voice:
Brand Personality:
Brand Colors:
Logo Description:

Target Audience:
Customer Pain Points:
Customer Aspirations:
Global Presence:

Certifications:
Partnerships:
Awards:
Media Mentions:
Public Sentiment Summary:

--------------------------------------------------

Then create a Dialogue & Sound Effects script.

IMPORTANT:
The script must match the user-requested:
- Video Length
- Audio Language

If user does not specify language → default English.
If user does not specify duration → default 30–45 seconds.

Script must feel natural, conversational, realistic.
No cinematic realism.
No animation references.

Output EXACTLY:

==================================================

Official Brand Name:

VIDEO_DURATION:
AUDIO_LANGUAGE:

--------------------------------------------------

BRAND_INTELLIGENCE_SUMMARY:

--------------------------------------------------

VIDEO_TITLE:
SCENE_DESCRIPTION:
SOUND_EFFECTS:
DIALOGUE:
CLOSING_LINE:

==================================================
""",
    tools=[google_search],
)

# ============================================================
# AGENT 2 — VIDEO DIRECTOR AGENT
# ============================================================

video_agent = LlmAgent(
    name="brand_dialogue_video_generator",
    model=TEXT_MODEL,
    instruction="""
You are a professional video generation director.

Take the ENTIRE structured output from the previous agent.

Extract the Official Brand Name from the structured output.

Pass the full structured output as prompt, and the brand name separately, to the generate_dialogue_video tool:

{
    "prompt": "<full structured output>",
    "brand_name": "<Official Brand Name>"
}

Do not modify content.
Do not shorten content.

After tool execution, clearly tell the user the generated filename.
""",
    tools=[generate_dialogue_video],
)

# ============================================================
# PIPELINE
# ============================================================

pipeline_agent = SequentialAgent(
    name="dynamic_brand_dialogue_video_pipeline",
    sub_agents=[research_agent, video_agent],
)

# ============================================================
# RUNNER
# ============================================================

async def main():

    session_service = InMemorySessionService()

    await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=SESSION_ID,
    )

    runner = Runner(
        agent=pipeline_agent,
        app_name=APP_NAME,
        session_service=session_service,
    )

    print("\n---Running Advanced Brand Dialogue Video Pipeline...---\n")

    # USER CAN SPECIFY LENGTH & LANGUAGE HERE
    user_prompt = """
    Create a dialogue-based brand introduction video for Infosys.
    Video length: 20 seconds.
    Audio language: Hindi.
    """

    user_message = types.Content(
        role="user",
        parts=[types.Part(text=user_prompt)]
    )

    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=user_message,
    ):
        if not event.content:
            continue

        for part in event.content.parts:

            if part.text:
                print("\n🧠 TEXT OUTPUT:\n", part.text)

            elif part.function_call:
                print("\n🔧 TOOL CALL →", part.function_call.name)

            elif part.function_response:
                print("\n✅ TOOL RESPONSE →", part.function_response.response)

    print("\n---Pipeline Complete.---\n")


# Run in Jupyter
await main()



---Running Advanced Brand Dialogue Video Pipeline...---


🧠 TEXT OUTPUT:

Official Brand Name: Infosys

VIDEO_DURATION: 20 seconds
AUDIO_LANGUAGE: Hindi

--------------------------------------------------

BRAND_INTELLIGENCE_SUMMARY:

Infosys Limited is an Indian multinational technology company that provides business consulting, information technology, and outsourcing services. Founded in 1981 by seven engineers with an initial capital of $250, it has grown into one of the largest IT companies in India, with headquarters in Bengaluru. As of March 2025, Infosys has 323,578 employees and operates in 59 countries.

Key aspects of Infosys' brand include:

*   **Services and Products:** Infosys offers a wide array of services, including software development, maintenance, and consulting in areas like digital experience, cloud computing, data analytics, and AI. Key products and platforms include Finacle, Panaya, Infosys Equinox, Infosys Meridian, and Infosys Cortex.
*   **Mission:** To ampl

In [9]:
"""
Dynamic Brand Introduction Video Generation Pipeline
Mode: Advanced Deep Research + Dialogue & Sound Effects (Veo 3.1)
User-Controlled Length & Language with Multilingual Support
"""

import os
import time
import asyncio
from datetime import datetime

from google import genai
from google.genai import types
from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.tools import FunctionTool
from google.adk.tools.google_search_tool import google_search
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from deep_translator import GoogleTranslator

from config import config

# ============================================================
# CONFIG
# ============================================================

os.environ["GOOGLE_API_KEY"] = config.GOOGLE_API_KEY

APP_NAME = "Dynamic-Brand-Dialogue-Video"
USER_ID = "user_1"
SESSION_ID = "session_1"

TEXT_MODEL = "gemini-2.5-flash"
VIDEO_MODEL = "veo-3.1-generate-preview"

# ============================================================
# TRANSLATION FUNCTION (POST-PROCESSING)
# ============================================================

async def translate_text(text: str, source: str, target: str) -> str:
    """
    Translate text from a source language to a target language using GoogleTranslator.
    """
    if text is None or source.lower() == target.lower():
        return text
    try:
        translator = GoogleTranslator(source=source, target=target)
        return translator.translate(text)
    except Exception as e:
        print(f"Translation error: {e}")
        return text

# ============================================================
# VIDEO GENERATION TOOL
# ============================================================

async def generate_dialogue_video_tool(prompt: str, brand_name: str) -> dict:
    """
    Generate a dialogue-based brand video using Veo 3.1.
    """
    client = genai.Client()

    print("\n🎬 Generating Dialogue & Sound Effects Brand Video...\n")

    operation = client.models.generate_videos(
        model=VIDEO_MODEL,
        prompt=prompt,
    )

    while not operation.done:
        print("⏳ Waiting for video generation...")
        time.sleep(10)
        operation = client.operations.get(operation)

    generated_video = operation.response.generated_videos[0]

    # Safe filename
    brand_name_clean = "".join(c if c.isalnum() else "_" for c in brand_name.strip())
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    os.makedirs("generated_videos", exist_ok=True)
    filename = os.path.join("generated_videos", f"{brand_name_clean}_{timestamp}.mp4")

    client.files.download(file=generated_video.video)
    generated_video.video.save(filename)

    print(f"\n✅ Video Generated Successfully: {filename}\n")

    return {
        "status": "success",
        "filename": filename,
    }

generate_dialogue_video = FunctionTool(func=generate_dialogue_video_tool)

# ============================================================
# AGENT 1 — RESEARCH AGENT (LIMIT RESEARCH CONTENT TO 4000-5000 WORDS)
# ============================================================

research_agent = LlmAgent(
    name="deep_brand_research_agent",
    model=TEXT_MODEL,
    instruction="""
You are an elite investigative brand intelligence analyst.

You MUST perform EXTENSIVE research using multiple Google searches.
Perform at least 12–15 search queries (official website, products, services, leadership, press, competitors, social media, etc.).
Collect detailed content, but **the total research summary must not exceed 4000–5000 words**.

Extract in detail:

Official Brand Name:
Founded Year:
Founder(s):
Headquarters:
Industry:
Website:

Products/Services:
USP:
Market Positioning:
Revenue Model (if available):

Mission:
Vision:
Core Values:
Brand Voice:
Brand Personality:
Brand Colors:
Logo Description:

Target Audience:
Customer Pain Points:
Customer Aspirations:
Global Presence:

Certifications:
Partnerships:
Awards:
Media Mentions:
Public Sentiment Summary:

Then create a Dialogue & Sound Effects script in the requested language.

IMPORTANT:
- Script must match the user-requested Video Length & Audio Language.
- If user does not specify language → default English.
- If user does not specify duration → default 30–45 seconds.

Output EXACTLY:

==================================================

Official Brand Name:

VIDEO_DURATION:
AUDIO_LANGUAGE:

--------------------------------------------------

BRAND_INTELLIGENCE_SUMMARY:

--------------------------------------------------

VIDEO_TITLE:
SCENE_DESCRIPTION:
SOUND_EFFECTS:
DIALOGUE:
CLOSING_LINE:

==================================================
""",
    tools=[google_search],  # removed translate_tool
)

# ============================================================
# AGENT 2 — VIDEO DIRECTOR AGENT
# ============================================================

video_agent = LlmAgent(
    name="brand_dialogue_video_generator",
    model=TEXT_MODEL,
    instruction="""
You are a professional video generation director.

Take the ENTIRE structured output from the previous agent.

If AUDIO_LANGUAGE is not English, use Python translation (outside the model) to translate
DIALOGUE, CLOSING_LINE, SCENE_DESCRIPTION from English to the target language.

Then pass the full structured output as prompt, and the brand name separately, to the generate_dialogue_video tool:

{
    "prompt": "<full structured output>",
    "brand_name": "<Official Brand Name>"
}

Do not modify content. Do not shorten content.

After tool execution, clearly tell the user the generated filename.
""",
    tools=[generate_dialogue_video],  # removed translate_tool
)

# ============================================================
# PIPELINE
# ============================================================

pipeline_agent = SequentialAgent(
    name="dynamic_brand_dialogue_video_pipeline",
    sub_agents=[research_agent, video_agent],
)

# ============================================================
# RUNNER
# ============================================================

async def main():
    session_service = InMemorySessionService()

    await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=SESSION_ID,
    )

    runner = Runner(
        agent=pipeline_agent,
        app_name=APP_NAME,
        session_service=session_service,
    )

    print("\n---Running Advanced Brand Dialogue Video Pipeline with Multilingual Support...---\n")

    # USER CAN SPECIFY LENGTH & LANGUAGE HERE
    user_prompt = """
    Create a dialogue-based brand introduction video for Infosys.
    Video length: 20 seconds.
    Audio language: Hindi.
    """

    user_message = types.Content(
        role="user",
        parts=[types.Part(text=user_prompt)]
    )

    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=user_message,
    ):
        if not event.content:
            continue

        for part in event.content.parts:

            if part.text:
                print("\n🧠 TEXT OUTPUT:\n", part.text)

                # Post-process translation if Hindi or any other language
                if "Audio language: Hindi" in part.text or "Audio language: hi" in part.text:
                    # Extract the script fields (DIALOGUE, CLOSING_LINE, SCENE_DESCRIPTION)
                    dialogue_start = part.text.find("DIALOGUE:")
                    closing_start = part.text.find("CLOSING_LINE:")
                    scene_start = part.text.find("SCENE_DESCRIPTION:")

                    if dialogue_start != -1 and closing_start != -1 and scene_start != -1:
                        dialogue_text = part.text[dialogue_start + len("DIALOGUE:"):closing_start].strip()
                        closing_text = part.text[closing_start + len("CLOSING_LINE:"):].strip()
                        scene_text = part.text[scene_start + len("SCENE_DESCRIPTION:"):dialogue_start].strip()  # optional

                        dialogue_text_hi = await translate_text(dialogue_text, "en", "hi")
                        closing_text_hi = await translate_text(closing_text, "en", "hi")
                        scene_text_hi = await translate_text(scene_text, "en", "hi") if scene_text else ""

                        print("\n📝 TRANSLATED SCRIPT (Hindi):\n")
                        print(f"SCENE_DESCRIPTION: {scene_text_hi}")
                        print(f"DIALOGUE: {dialogue_text_hi}")
                        print(f"CLOSING_LINE: {closing_text_hi}")

            elif part.function_call:
                print("\n🔧 TOOL CALL →", part.function_call.name)

            elif part.function_response:
                print("\n✅ TOOL RESPONSE →", part.function_response.response)

    print("\n---Pipeline Complete.---\n")


# Run in Jupyter
await main()


---Running Advanced Brand Dialogue Video Pipeline with Multilingual Support...---


🧠 TEXT OUTPUT:

Official Brand Name: Infosys Limited

VIDEO_DURATION: 20 seconds
AUDIO_LANGUAGE: Hindi

--------------------------------------------------

BRAND_INTELLIGENCE_SUMMARY:

Infosys Limited, founded in 1981 by seven engineers including N. R. Narayana Murthy, is an Indian multinational technology company headquartered in Bengaluru, Karnataka. It operates in the Information Technology, Business Consulting, and Outsourcing Services industry. Its official website is infosys.com.

**Products/Services:** Infosys offers a comprehensive suite of IT services including application development, maintenance, modernization, systems integration, engineering services, infrastructure management, and business process outsourcing (BPO). A key focus is on next-generation digital services such as Cloud, Artificial Intelligence (AI), Internet of Things (IoT), and Cybersecurity. The company also provides propriet

In [19]:
"""
Dynamic Brand Introduction Video Generation Pipeline
Mode: Advanced Deep Research + Dialogue & Sound Effects (Veo 3.1)
User-Controlled Length, Language, Aspect Ratio & Resolution with Multilingual Support

CORRECT FLOW:
1. User specifies brand name, video length, audio language, aspect ratio, resolution
2. Research Agent collects brand intelligence in English (4000-5000 words)
   and writes ALL fields in English — including SCENE_DESCRIPTION (camera/text directions
   always stay English so Veo understands them)
3. Translation Step (only when language ≠ English):
     - Translates ONLY → DIALOGUE and CLOSING_LINE (spoken/voiceover lines)
     - Keeps SCENE_DESCRIPTION in English (so on-screen text stays English)
     - Splits large fields into ≤ 1500-char chunks before translating
4. Video Agent builds a focused video prompt (not the full research dump) and
   passes it to Veo 3.1 with the correct aspect_ratio and resolution.
   The prompt explicitly states the required duration so Veo respects it.

KEY FIXES vs previous version
──────────────────────────────
* On-screen text stays English  → SCENE_DESCRIPTION is never translated
* Video length respected        → duration is embedded as a strong instruction
                                  inside the prompt, not just in metadata
* Skip translation for English  → early-exit when target lang == "en"
* Cleaner video prompt          → only VIDEO_TITLE, SCENE_DESCRIPTION, SOUND_EFFECTS,
                                  DIALOGUE (translated), CLOSING_LINE (translated)
                                  are sent to Veo — not the 4000-word research dump
"""

import os
import re
import time
import asyncio
from datetime import datetime

from google import genai
from google.genai import types
from google.adk.agents import LlmAgent
from google.adk.tools import FunctionTool
from google.adk.tools.google_search_tool import google_search
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from deep_translator import GoogleTranslator

from config import config

# ============================================================
# GLOBAL CONFIG
# ============================================================

os.environ["GOOGLE_API_KEY"] = config.GOOGLE_API_KEY

APP_NAME       = "Dynamic-Brand-Dialogue-Video"
USER_ID        = "user_1"
SESSION_ID_RES = "session_research"
SESSION_ID_VID = "session_video"

TEXT_MODEL  = "gemini-2.5-flash"
VIDEO_MODEL = "veo-3.1-generate-preview"

# ============================================================
# LANGUAGE CODE MAP  (33 major languages)
# ============================================================
# Maps human-readable language names → ISO 639-1 / BCP-47 codes
# accepted by deep_translator (GoogleTranslator).
#
#   english      → en     hindi       → hi     spanish     → es
#   french       → fr     german      → de     arabic      → ar
#   chinese      → zh-CN  japanese    → ja     korean      → ko
#   portuguese   → pt     italian     → it     russian     → ru
#   dutch        → nl     turkish     → tr     polish      → pl
#   swedish      → sv     norwegian   → no     danish      → da
#   finnish      → fi     greek       → el     hebrew      → iw
#   thai         → th     vietnamese  → vi     indonesian  → id
#   malay        → ms     tagalog     → tl     swahili     → sw
#   bengali      → bn     punjabi     → pa     urdu        → ur
#   tamil        → ta     telugu      → te     marathi     → mr
#
# For any unlisted language pass its BCP-47 code directly.
# ============================================================

LANGUAGE_CODE_MAP: dict[str, str] = {
    "english":    "en",
    "hindi":      "hi",
    "spanish":    "es",
    "french":     "fr",
    "german":     "de",
    "arabic":     "ar",
    "chinese":    "zh-CN",
    "japanese":   "ja",
    "korean":     "ko",
    "portuguese": "pt",
    "italian":    "it",
    "russian":    "ru",
    "dutch":      "nl",
    "turkish":    "tr",
    "polish":     "pl",
    "swedish":    "sv",
    "norwegian":  "no",
    "danish":     "da",
    "finnish":    "fi",
    "greek":      "el",
    "hebrew":     "iw",
    "thai":       "th",
    "vietnamese": "vi",
    "indonesian": "id",
    "malay":      "ms",
    "tagalog":    "tl",
    "swahili":    "sw",
    "bengali":    "bn",
    "punjabi":    "pa",
    "urdu":       "ur",
    "tamil":      "ta",
    "telugu":     "te",
    "marathi":    "mr",
}

# ============================================================
# VALID VIDEO CONFIG OPTIONS
# ============================================================

VALID_ASPECT_RATIOS = {"16:9", "9:16", "1:1", "4:3", "3:4"}
VALID_RESOLUTIONS   = {"720p", "1080p", "4k"}

# ============================================================
# HELPERS: structured-field extraction / replacement
# ============================================================

# Stop-pattern: any known field heading, section divider, or end-of-string
_STOP = (
    r"(?=\n(?:"
    r"OFFICIAL[ _]BRAND[ _]NAME|VIDEO_DURATION|AUDIO_LANGUAGE|ASPECT_RATIO|RESOLUTION"
    r"|VIDEO_TITLE|SCENE_DESCRIPTION|SOUND_EFFECTS|DIALOGUE|CLOSING_LINE"
    r"|BRAND_INTELLIGENCE_SUMMARY"
    r"):|={5,}|-{5,}|\Z)"
)


def extract_field(text: str, field_name: str) -> str:
    """Return the content of a named field (case-insensitive)."""
    pattern = rf"(?i){re.escape(field_name)}:\s*(.*?){_STOP}"
    m = re.search(pattern, text, re.DOTALL)
    return m.group(1).strip() if m else ""


def replace_field(text: str, field_name: str, new_value: str) -> str:
    """Replace the content of a named field in-place."""
    pattern = rf"(?i)({re.escape(field_name)}:\s*)(.*?){_STOP}"
    return re.sub(pattern, rf"\g<1>{new_value}\n", text, flags=re.DOTALL)


def get_language_code(language_name: str) -> str:
    """
    Map a human-readable language name to its translation code.
    Falls back to the raw string (lower-cased) when not found.
    """
    return LANGUAGE_CODE_MAP.get(language_name.strip().lower(), language_name.strip().lower())


# ============================================================
# CHUNKED TRANSLATION  (avoids GoogleTranslator character limits)
# ============================================================

CHUNK_SIZE = 1500   # characters per chunk


def split_into_chunks(text: str, chunk_size: int = CHUNK_SIZE) -> list[str]:
    """
    Split text into chunks of ≤ chunk_size characters,
    breaking only at line boundaries to preserve meaning.
    """
    chunks:  list[str] = []
    lines              = text.splitlines(keepends=True)
    current            = ""

    for line in lines:
        if len(current) + len(line) > chunk_size and current:
            chunks.append(current)
            current = line
        else:
            current += line

    if current:
        chunks.append(current)

    return chunks


async def translate_chunks(text: str, source: str, target: str) -> str:
    """
    Translate text safely by:
      1. Splitting into ≤ CHUNK_SIZE char chunks at line boundaries
      2. Translating each chunk independently
      3. Joining results back into one string

    Keeps the original chunk if translation of that piece fails.
    """
    if not text or source == target:
        return text

    chunks            = split_into_chunks(text, CHUNK_SIZE)
    translated_chunks: list[str] = []

    print(f"   📦 {len(chunks)} chunk(s) to translate...")

    for idx, chunk in enumerate(chunks, 1):
        try:
            translator = GoogleTranslator(source=source, target=target)
            result     = translator.translate(chunk)
            translated_chunks.append(result or chunk)
            print(f"   ✅ Chunk {idx}/{len(chunks)} done.")
        except Exception as e:
            print(f"   ⚠️  Chunk {idx}/{len(chunks)} failed ({e}). Keeping original.")
            translated_chunks.append(chunk)

    return "".join(translated_chunks)


# ============================================================
# TRANSLATION STEP
# ============================================================

async def translate_script_fields(structured_output: str) -> str:
    """
    Translates ONLY the spoken/voiceover fields into the target language:
      ✅ DIALOGUE      — voiceover lines spoken in the video  → TRANSLATED
      ✅ CLOSING_LINE  — final spoken tagline                 → TRANSLATED
      ❌ SCENE_DESCRIPTION — camera & text-overlay directions → KEPT IN ENGLISH
                             (on-screen text must stay English so Veo renders
                              readable English text in the video)
      ❌ SOUND_EFFECTS — technical sound cues                 → KEPT IN ENGLISH
      ❌ BRAND_INTELLIGENCE_SUMMARY — brand facts             → KEPT IN ENGLISH

    Skips translation entirely when AUDIO_LANGUAGE is English (or unspecified).
    """
    audio_language_raw = extract_field(structured_output, "AUDIO_LANGUAGE").strip()
    target_lang_code   = get_language_code(audio_language_raw)

    # ── Early exit for English ──────────────────────────────────
    if not audio_language_raw or target_lang_code == "en":
        print("\n🌐 Audio language is English — skipping translation.\n")
        return structured_output

    print(f"\n🌐 Target language : '{audio_language_raw}' (code: {target_lang_code})")
    print(f"   Chunk size      : {CHUNK_SIZE} chars")
    print(f"   Fields to translate : DIALOGUE, CLOSING_LINE")
    print(f"   Fields kept in English : SCENE_DESCRIPTION, SOUND_EFFECTS\n")

    # ── Translate ONLY spoken fields ────────────────────────────
    for field in ["DIALOGUE", "CLOSING_LINE"]:
        original = extract_field(structured_output, field)
        if not original:
            print(f"⚠️  Could not extract '{field}' — skipping.")
            continue

        print(f"🔤 Translating {field} ({len(original)} chars)...")
        translated        = await translate_chunks(original, source="en", target=target_lang_code)
        structured_output = replace_field(structured_output, field, translated)
        print(f"   ✅ {field} complete.\n")

    return structured_output


# ============================================================
# VIDEO PROMPT BUILDER
# ============================================================

def build_video_prompt(structured_output: str) -> str:
    """
    Builds a focused, Veo-optimised prompt from the structured research output.

    IMPORTANT DESIGN DECISIONS:
    ─────────────────────────────
    1. Duration is embedded as a strong explicit instruction at the top of the
       prompt (e.g. "Create a 20-second video..."). Veo 3.1 does NOT accept a
       duration parameter in GenerateVideosConfig — it must be in the prompt text.

    2. Only the script fields are sent to Veo (VIDEO_TITLE, SCENE_DESCRIPTION,
       SOUND_EFFECTS, DIALOGUE, CLOSING_LINE). The 4000-word
       BRAND_INTELLIGENCE_SUMMARY is excluded — it confuses Veo's attention and
       does not contribute to video quality.

    3. SCENE_DESCRIPTION is always English (camera directions, timing, text
       overlays) so Veo renders English on-screen text.

    4. DIALOGUE / CLOSING_LINE contain the translated voiceover so Veo generates
       audio in the target language.
    """
    duration       = extract_field(structured_output, "VIDEO_DURATION").strip() or "30"
    audio_language = extract_field(structured_output, "AUDIO_LANGUAGE").strip() or "English"
    video_title    = extract_field(structured_output, "VIDEO_TITLE")
    scene_desc     = extract_field(structured_output, "SCENE_DESCRIPTION")
    sound_fx       = extract_field(structured_output, "SOUND_EFFECTS")
    dialogue       = extract_field(structured_output, "DIALOGUE")
    closing_line   = extract_field(structured_output, "CLOSING_LINE")

    prompt = f"""Create a professional brand introduction video that is exactly {duration} seconds long.

VIDEO TITLE: {video_title}

DURATION INSTRUCTION:
This video must be exactly {duration} seconds in total length. Pace all scenes accordingly.

AUDIO LANGUAGE:
All spoken dialogue and voiceover must be delivered in {audio_language}.
On-screen text overlays and captions must remain in English.

SCENE DESCRIPTION (camera directions and timing — follow exactly):
{scene_desc}

SOUND EFFECTS:
{sound_fx}

DIALOGUE (spoken voiceover in {audio_language}):
{dialogue}

CLOSING LINE (final spoken line in {audio_language}):
{closing_line}
"""
    return prompt.strip()


# ============================================================
# VIDEO GENERATION TOOL
# ============================================================

async def generate_dialogue_video_tool(
    prompt:       str,
    brand_name:   str,
    aspect_ratio: str = "16:9",
    resolution:   str = "1080p",
) -> dict:
    """
    Generate a dialogue-based brand video using Veo 3.1.

    Parameters
    ----------
    prompt : str
        Focused video generation prompt (built by build_video_prompt).
        Already contains the duration instruction and translated dialogue.
    brand_name : str
        Official brand name — used for the output filename only.
    aspect_ratio : str
        Frame shape of the generated video.
        Valid values:
          "16:9"  → Landscape / widescreen  (default — TV, YouTube)
          "9:16"  → Portrait / vertical     (Reels, Shorts, TikTok)
          "1:1"   → Square                  (Instagram feed)
          "4:3"   → Classic TV
          "3:4"   → Portrait TV
    resolution : str
        Output resolution.
        Valid values:
          "720p"  → HD       (fastest generation, smallest file)
          "1080p" → Full HD  (default — balanced quality)
          "4k"    → Ultra HD (best quality, slowest generation)

    Language code quick reference (for AUDIO_LANGUAGE in prompt)
    ──────────────────────────────────────────────────────────────
    english → en     hindi → hi      spanish  → es     french   → fr
    german  → de     arabic → ar     chinese  → zh-CN  japanese → ja
    korean  → ko     portuguese → pt italian  → it     russian  → ru
    dutch   → nl     turkish → tr    polish   → pl     swedish  → sv
    norwegian → no   danish → da     finnish  → fi     greek    → el
    hebrew  → iw     thai → th       vietnamese → vi   indonesian → id
    malay   → ms     tagalog → tl    swahili  → sw     bengali  → bn
    punjabi → pa     urdu → ur       tamil    → ta     telugu   → te
    marathi → mr
    """
    # Validate and normalise inputs
    if aspect_ratio not in VALID_ASPECT_RATIOS:
        print(f"Invalid aspect_ratio '{aspect_ratio}' — falling back to '16:9'.")
        aspect_ratio = "16:9"
    if resolution not in VALID_RESOLUTIONS:
        print(f"Invalid resolution '{resolution}' — falling back to '1080p'.")
        resolution = "1080p"

    client = genai.Client()

    print(f"\n🎬 Generating Brand Video...")
    print(f"   Aspect Ratio : {aspect_ratio}")
    print(f"   Resolution   : {resolution}")
    print(f"\n📋 Video Prompt:\n{'-'*50}\n{prompt}\n{'-'*50}\n")

    operation = client.models.generate_videos(
        model=VIDEO_MODEL,
        prompt=prompt,
        config=types.GenerateVideosConfig(
            aspect_ratio=aspect_ratio,
            resolution=resolution,
        ),
    )

    while not operation.done:
        print("⏳ Waiting for video generation...")
        time.sleep(10)
        operation = client.operations.get(operation)

    generated_video = operation.response.generated_videos[0]

    # Safe filename
    brand_clean = "".join(c if c.isalnum() else "_" for c in brand_name.strip())
    timestamp   = datetime.now().strftime("%Y%m%d_%H%M%S")
    os.makedirs("generated_videos", exist_ok=True)
    filename = os.path.join("generated_videos", f"{brand_clean}_{timestamp}.mp4")

    client.files.download(file=generated_video.video)
    generated_video.video.save(filename)

    print(f"\n✅ Video Saved: {filename}\n")

    return {
        "status":       "success",
        "filename":     filename,
        "aspect_ratio": aspect_ratio,
        "resolution":   resolution,
    }

generate_dialogue_video = FunctionTool(func=generate_dialogue_video_tool)


# ============================================================
# AGENT 1 — RESEARCH AGENT
# ============================================================

research_agent = LlmAgent(
    name="deep_brand_research_agent",
    model=TEXT_MODEL,
    instruction="""
You are an elite investigative brand intelligence analyst.

Perform EXTENSIVE research using Google Search.
Run at least 12–15 queries covering: official website, products, services, leadership,
press releases, competitors, social media, ESG/CSR, financial results, partnerships, etc.

Collect detailed content but keep the BRAND_INTELLIGENCE_SUMMARY within 2000–3000 words.

Extract all fields listed below:

Official Brand Name:
Founded Year:
Founder(s):
Headquarters:
Industry:
Website:
Products/Services:
USP:
Market Positioning:
Revenue Model:
Mission:
Vision:
Core Values:
Brand Voice:
Brand Personality:
Brand Colors:
Logo Description:
Target Audience:
Customer Pain Points:
Customer Aspirations:
Global Presence:
Certifications:
Partnerships:
Awards:
Media Mentions:
Public Sentiment Summary:

Then write the video script.

CRITICAL RULES — READ CAREFULLY:

1. Write ALL fields — including SCENE_DESCRIPTION, DIALOGUE, and CLOSING_LINE — in
   ENGLISH ONLY. Do NOT write any foreign-language text anywhere.
   A dedicated translation step runs after you and handles spoken dialogue only.

2. In SCENE_DESCRIPTION:
   - Write all camera directions, timing cues, and text overlay instructions in English.
   - Text overlay instructions must say things like:
       Text overlay: "Your digital transformation partner"
     NOT in Hindi or any other language.
   - This ensures on-screen text renders in English in the final video.

3. In DIALOGUE:
   - Write all voiceover/narration lines in English only.
   - These lines will be translated into the target language after you finish.

4. Record VIDEO_DURATION, AUDIO_LANGUAGE, ASPECT_RATIO, and RESOLUTION exactly
   as the user specified.
   - If the user did not specify language      → write: English
   - If the user did not specify duration      → write: 30
   - If the user did not specify aspect_ratio  → write: 16:9
   - If the user did not specify resolution    → write: 1080p

Output EXACTLY this format (no deviations, no extra sections):

==================================================

Official Brand Name:

VIDEO_DURATION:
AUDIO_LANGUAGE:
ASPECT_RATIO:
RESOLUTION:

--------------------------------------------------

BRAND_INTELLIGENCE_SUMMARY:

--------------------------------------------------

VIDEO_TITLE:
SCENE_DESCRIPTION:
SOUND_EFFECTS:
DIALOGUE:
CLOSING_LINE:

==================================================
""",
    tools=[google_search],
)


# ============================================================
# AGENT 2 — VIDEO DIRECTOR AGENT
# ============================================================

video_agent = LlmAgent(
    name="brand_dialogue_video_generator",
    model=TEXT_MODEL,
    instruction="""
You are a professional video generation director.

You will receive a fully prepared video prompt and brand metadata.
The prompt already contains:
  - The correct video duration as an explicit instruction
  - The scene description in English (for on-screen text)
  - The translated dialogue in the target language (for voiceover audio)

Your ONLY job:
1. Extract from the message:
     - "prompt"       → the video generation prompt (the full block labelled VIDEO PROMPT)
     - "brand_name"   → from the BRAND_NAME field
     - "aspect_ratio" → from the ASPECT_RATIO field
     - "resolution"   → from the RESOLUTION field

2. Call generate_dialogue_video_tool with all four parameters exactly as provided.
   Do NOT modify, summarise, or shorten the prompt.

3. After the tool call completes, report to the user:
     - Output filename
     - Aspect ratio used
     - Resolution used
""",
    tools=[generate_dialogue_video],
)


# ============================================================
# PIPELINE ORCHESTRATOR
# ============================================================

async def run_pipeline(user_prompt: str) -> None:
    """
    Full three-step pipeline:

      Step 1 → Research Agent
               Collects brand data and writes all script fields in English.

      Step 2 → Chunked Translation
               Translates ONLY DIALOGUE and CLOSING_LINE into the target language.
               SCENE_DESCRIPTION stays in English (keeps on-screen text English).
               Skipped entirely when AUDIO_LANGUAGE is English.

      Step 3 → Video Prompt Builder + Video Generation Agent
               Builds a focused Veo prompt (no research dump) with the duration
               embedded as an explicit instruction, then generates the video.
    """
    session_service = InMemorySessionService()

    # ── STEP 1: Research ────────────────────────────────────────
    print("\n" + "="*60)
    print("  STEP 1: Brand Research Agent")
    print("="*60 + "\n")

    await session_service.create_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID_RES
    )

    research_runner = Runner(
        agent=research_agent,
        app_name=APP_NAME,
        session_service=session_service,
    )

    research_output = ""
    async for event in research_runner.run_async(
        user_id=USER_ID,
        session_id=SESSION_ID_RES,
        new_message=types.Content(role="user", parts=[types.Part(text=user_prompt)]),
    ):
        if not event.content:
            continue
        for part in event.content.parts:
            if part.text:
                research_output += part.text
                print("\n📊 RESEARCH OUTPUT:\n", part.text)
            elif part.function_call:
                print(f"\n🔍 SEARCH → {part.function_call.args}")
            elif part.function_response:
                print(f"\n📥 RESULT (truncated) → {str(part.function_response.response)[:200]}...")

    if not research_output:
        print("❌ Research agent produced no output. Aborting.")
        return

    # ── STEP 2: Translate DIALOGUE & CLOSING_LINE only ──────────
    print("\n" + "="*60)
    print("  STEP 2: Chunked Translation  (DIALOGUE + CLOSING_LINE only)")
    print("="*60 + "\n")

    translated_output = await translate_script_fields(research_output)

    # ── STEP 3: Build focused prompt → Generate Video ────────────
    print("\n" + "="*60)
    print("  STEP 3: Video Prompt Builder + Video Generation Agent")
    print("="*60 + "\n")

    # Extract config values for the video agent message
    brand_name   = extract_field(translated_output, "Official Brand Name") or "Brand"
    aspect_ratio = extract_field(translated_output, "ASPECT_RATIO")        or "16:9"
    resolution   = extract_field(translated_output, "RESOLUTION")          or "1080p"

    # Build a clean, focused video prompt (no research dump sent to Veo)
    video_prompt = build_video_prompt(translated_output)

    print("\n📋 FINAL VIDEO PROMPT SENT TO VEO:\n")
    print(video_prompt)
    print()

    await session_service.create_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID_VID
    )

    video_runner = Runner(
        agent=video_agent,
        app_name=APP_NAME,
        session_service=session_service,
    )

    # Package prompt + config neatly for the video agent
    video_agent_message = (
        f"BRAND_NAME: {brand_name}\n"
        f"ASPECT_RATIO: {aspect_ratio}\n"
        f"RESOLUTION: {resolution}\n\n"
        f"VIDEO PROMPT:\n{video_prompt}"
    )

    async for event in video_runner.run_async(
        user_id=USER_ID,
        session_id=SESSION_ID_VID,
        new_message=types.Content(role="user", parts=[types.Part(text=video_agent_message)]),
    ):
        if not event.content:
            continue
        for part in event.content.parts:
            if part.text:
                print("\n🎬 VIDEO AGENT OUTPUT:\n", part.text)
            elif part.function_call:
                args = part.function_call.args
                print(f"\n🔧 TOOL CALL → {part.function_call.name}")
                print(f"   brand_name   : {args.get('brand_name',   'N/A')}")
                print(f"   aspect_ratio : {args.get('aspect_ratio', 'N/A')}")
                print(f"   resolution   : {args.get('resolution',   'N/A')}")
            elif part.function_response:
                print(f"\n✅ TOOL RESPONSE → {part.function_response.response}")

    print("\n" + "="*60)
    print("  Pipeline Complete.")
    print("="*60 + "\n")


# ============================================================
# ENTRY POINT
# ============================================================

async def main() -> None:
    # ── USER INPUT ─────────────────────────────────────────────
    # brand        : any company / product name
    # Video length : duration in seconds  (Veo respects this via prompt instruction)
    # Audio language: spoken voiceover language (on-screen text always English)
    #   options    : see LANGUAGE_CODE_MAP above
    # Aspect ratio : "16:9" | "9:16" | "1:1" | "4:3" | "3:4"
    # Resolution   : "720p" | "1080p" | "4k"
    # ──────────────────────────────────────────────────────────
    user_prompt = """
    Create a dialogue-based brand introduction video for Philips.
    Audio language : Hindi
    Aspect ratio   : 16:9
    Resolution     : 720p
    """

    await run_pipeline(user_prompt)


# Run in Jupyter:
await main()

# Run in terminal (uncomment and comment the line above):
# if __name__ == "__main__":
#     asyncio.run(main())


  STEP 1: Brand Research Agent


📊 RESEARCH OUTPUT:
 Official Brand Name: Koninklijke Philips N.V. (branded Philips)

VIDEO_DURATION: 60
AUDIO_LANGUAGE: Hindi
ASPECT_RATIO: 16:9
RESOLUTION: 720p

--------------------------------------------------

BRAND_INTELLIGENCE_SUMMARY:
Philips, officially Koninklijke Philips N.V., is a Dutch multinational health technology company founded in Eindhoven in 1891 by Gerard Philips and his father Frederik. Headquartered in Amsterdam, Philips has transformed from its origins in light bulbs and consumer electronics into a global leader focused on improving people's health and well-being through meaningful innovation.

The company's core **products and services** span the health continuum, from healthy living and prevention to diagnosis, treatment, and home care. This includes advanced diagnostic imaging (MRI, CT, ultrasound), image-guided therapy, patient monitoring, connected care solutions, and health informatics for healthcare professionals. For con

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED'}}